In [ ]:
import os
import torch
from tqdm import tqdm
# os.environ["CUDA_VISIBLE_DEVICES"] = "7"

In [ ]:
from data import load_json_dataset

path = "hotel_dataset/hotel_aste_test_augmented.json"
dataset = load_json_dataset(path)

In [ ]:
from huggingface_hub import login
login(token='YOUR_HF_TOKEN')

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
checkpoint_path = "absa-research/hotel_aste_train_augmented_noreasoning_model-Qwen2.5-0.5B_lr-0.0001_bs-16_epochs-20"
model = AutoModelForCausalLM.from_pretrained(checkpoint_path, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(checkpoint_path)

In [5]:
# # Load dataset from huggingface "absa-research/counterfact_data"
# from datasets import load_dataset
# dataset = load_dataset("absa-research/counterfact_data")

In [ ]:
from preprocess import preprocess_function, preprocess_function_base

preprocess_fn = lambda examples: preprocess_function_base(examples, tokenizer)
tokenized_dataset = dataset.map(preprocess_fn, batched=True)

In [ ]:
def generate_in_batches(model, tokenized_dataset, tokenizer, batch_size=16, device='cuda', max_length=1024, do_sample=False, logits_processor=None):
	predictions = []
	
	# Calculate number of batches
	num_samples = len(tokenized_dataset["input_ids"])
	num_batches = (num_samples + batch_size - 1) // batch_size  # Ceiling division
	
	for batch_idx in tqdm(range(num_batches)):
		# Calculate start and end indices for the current batch
		start_idx = batch_idx * batch_size
		end_idx = min((batch_idx + 1) * batch_size, num_samples)
		
		# Get batch of input IDs and labels
		batch_input_ids = torch.tensor(tokenized_dataset["input_ids"][start_idx:end_idx])
		batch_labels = torch.tensor(tokenized_dataset["labels"][start_idx:end_idx])
		prefix_len = (batch_labels == -100).sum(axis=1)
		
		# Extract user prompts (excluding assistant responses) using labels
		user_prompts = []
		for i in range(len(batch_input_ids)):
			user_prompts.append(batch_input_ids[i][:prefix_len[i].item()])
		
		# Pad user prompts to same length
		max_prompt_len = max(len(prompt) for prompt in user_prompts)
		padded_prompts = []
		prompt_attention_masks = []

		for prompt in user_prompts:
			# Calculate padding needed
			padding_length = max_prompt_len - len(prompt)
			
			# Left padding: padding tokens first, then actual tokens
			padded_prompts.append([tokenizer.pad_token_id] * padding_length + prompt.tolist())
			prompt_attention_masks.append([0] * padding_length + [1] * len(prompt))		
		
		# Convert to tensors
		padded_prompts_tensor = torch.tensor(padded_prompts)
		prompt_attention_masks_tensor = torch.tensor(prompt_attention_masks)
		
		# Generate outputs
		batch_outputs = model.generate(
			input_ids=padded_prompts_tensor.to(device),
			attention_mask=prompt_attention_masks_tensor.to(device),
			max_length=max_length,
			do_sample=do_sample,
			pad_token_id=tokenizer.pad_token_id,
			logits_processor=logits_processor, # List of processors to apply
		)
		
		# Decode and append each output in the batch
		for i, output_ids in enumerate(batch_outputs):
			# Extract only the generated part (after the prompt)
			prompt_len = len(user_prompts[i])
			padding_length = max_prompt_len - prompt_len
			generated_ids = output_ids[padding_length + prompt_len:]
			decoded_output = tokenizer.decode(generated_ids, skip_special_tokens=True)
			predictions.append(decoded_output)
		# break
	
	return predictions

In [ ]:
predictions = generate_in_batches(
	model,
	tokenized_dataset,
	tokenizer,
	batch_size=256,
	device='cuda',
	max_length=1024,
	# do_sample=True
)

In [ ]:
import json
output_file = "predictions.json"
with open(output_file, 'w') as f:
	json.dump(predictions, f, indent=4)

In [ ]:
import json
predictions = []
with open("predictions.json", 'r') as f:
	predictions = json.load(f)
print(len(predictions))